# Probe design demos

**Download notebook:** [mining_demos.ipynb](mining_demos.ipynb)

Demonstrates the core OligoMiner2 probe design workflow: mining candidate probes
from a FASTA file, customizing mining parameters, building alignment and kmer
indexes, and computing sequence properties.

### configure notebook

In [ ]:
%reload_ext autoreload
%autoreload 2

from oligominer import mine_fasta, get_example_fasta_path, ProbeSet
from oligominer.utils import seq_utils
from oligominer.specificity.alignment import bowtie_build
from oligominer.specificity.kmers import jellyfish_build



print('DONE!')

DONE!


### mine initial candidate sequences from input fasta

In [3]:
# locate example FASTA bundled with the package
EXAMPLE_FASTA = get_example_fasta_path()

# mine candidate probes with default parameters
ps = ProbeSet(mine_fasta(EXAMPLE_FASTA))

print(f'DONE! Mined {len(ps)} candidate probes.')
ps.df

DONE! Mined 14328 candidate probes.


,seq_id,start,stop,probe_seq,tm,seqid
0,chrI,0,30,CCACACCACACCCACACACCCACACACCAC,46.06,chrI:0-30
1,chrI,1,31,CACACCACACCCACACACCCACACACCACA,45.69,chrI:1-31
2,chrI,2,32,ACACCACACCCACACACCCACACACCACAC,45.69,chrI:2-32
3,chrI,3,33,CACCACACCCACACACCCACACACCACACC,46.06,chrI:3-33
4,chrI,4,34,ACCACACCCACACACCCACACACCACACCA,46.74,chrI:4-34
...,...,...,...,...,...,...
14323,chrIV,14817,14854,CATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.63,chrIV:14817-14854
14324,chrIV,14818,14854,ATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.29,chrIV:14818-14854
14325,chrIV,14819,14854,TACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.31,chrIV:14819-14854
14326,chrIV,14820,14854,ACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.75,chrIV:14820-14854


### customizing mining parameters

Mining parameters such as probe length, Tm range, and GC content are passed
directly to `mine_fasta()`. See the API reference for the full list of options.

In [4]:
# mine with custom Tm and length constraints
ps_custom = ProbeSet(mine_fasta(EXAMPLE_FASTA, min_length=36, max_length=45, min_tm=40, max_tm=50))

print(f'DONE! Mined {len(ps_custom)} candidate probes.')
ps_custom.df

DONE! Mined 36524 candidate probes.


,seq_id,start,stop,probe_seq,tm,seqid
0,chrI,0,36,CCACACCACACCCACACACCCACACACCACACCACA,49.85,chrI:0-36
1,chrI,1,37,CACACCACACCCACACACCCACACACCACACCACAC,48.95,chrI:1-37
2,chrI,2,38,ACACCACACCCACACACCCACACACCACACCACACA,49.56,chrI:2-38
3,chrI,3,39,CACCACACCCACACACCCACACACCACACCACACAC,48.95,chrI:3-39
4,chrI,4,40,ACCACACCCACACACCCACACACCACACCACACACC,49.86,chrI:4-40
...,...,...,...,...,...,...
36519,chrIV,14860,14900,ATACCAGGAGAATTCCTTGTCTAATTTGCTACAAGTACCT,40.11,chrIV:14860-14900
36520,chrIV,14861,14900,TACCAGGAGAATTCCTTGTCTAATTTGCTACAAGTACCT,40.08,chrIV:14861-14900
36521,chrIV,14862,14900,ACCAGGAGAATTCCTTGTCTAATTTGCTACAAGTACCT,40.42,chrIV:14862-14900
36522,chrIV,14863,14904,CCAGGAGAATTCCTTGTCTAATTTGCTACAAGTACCTTTTC,40.18,chrIV:14863-14904


In [5]:
# mine with a narrow length window
ps_narrow = ProbeSet(mine_fasta(EXAMPLE_FASTA, min_length=38, max_length=40))

print(f'DONE! Mined {len(ps_narrow)} candidate probes.')
ps_narrow.df

DONE! Mined 17370 candidate probes.


,seq_id,start,stop,probe_seq,tm,seqid
0,chrI,31,69,CCACACACCACACCACACCCACACACACACATCCTAAC,46.91,chrI:31-69
1,chrI,32,70,CACACACCACACCACACCCACACACACACATCCTAACA,46.61,chrI:32-70
2,chrI,33,71,ACACACCACACCACACCCACACACACACATCCTAACAC,46.62,chrI:33-71
3,chrI,34,72,CACACCACACCACACCCACACACACACATCCTAACACT,46.49,chrI:34-72
4,chrI,35,73,ACACCACACCACACCCACACACACACATCCTAACACTA,45.82,chrI:35-73
...,...,...,...,...,...,...
17365,chrIV,14817,14855,CATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCTA,42.23,chrIV:14817-14855
17366,chrIV,14818,14857,ATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCTAAC,42.30,chrIV:14818-14857
17367,chrIV,14819,14857,TACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCTAAC,42.32,chrIV:14819-14857
17368,chrIV,14820,14858,ACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCTAACT,43.39,chrIV:14820-14858


### compute some sequence properties

In [6]:
# compute some derived sequence properties
ps.df['length'] = ps.df['probe_seq'].apply(len)
ps.df['gc'] = ps.df['probe_seq'].apply(seq_utils.calc_gc)
ps.df['rc_seq'] = ps.df['probe_seq'].apply(seq_utils.rev_comp)

ps.df

,seq_id,start,stop,probe_seq,tm,seqid,length,gc,rc_seq
0,chrI,0,30,CCACACCACACCCACACACCCACACACCAC,46.06,chrI:0-30,30,0.633333,GTGGTGTGTGGGTGTGTGGGTGTGGTGTGG
1,chrI,1,31,CACACCACACCCACACACCCACACACCACA,45.69,chrI:1-31,30,0.600000,TGTGGTGTGTGGGTGTGTGGGTGTGGTGTG
2,chrI,2,32,ACACCACACCCACACACCCACACACCACAC,45.69,chrI:2-32,30,0.600000,GTGTGGTGTGTGGGTGTGTGGGTGTGGTGT
3,chrI,3,33,CACCACACCCACACACCCACACACCACACC,46.06,chrI:3-33,30,0.633333,GGTGTGGTGTGTGGGTGTGTGGGTGTGGTG
4,chrI,4,34,ACCACACCCACACACCCACACACCACACCA,46.74,chrI:4-34,30,0.600000,TGGTGTGGTGTGTGGGTGTGTGGGTGTGGT
...,...,...,...,...,...,...,...,...,...
14323,chrIV,14817,14854,CATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.63,chrIV:14817-14854,37,0.459459,AGAGAAACAGAAATATGAAACGGCCACCCCAGGTATG
14324,chrIV,14818,14854,ATACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.29,chrIV:14818-14854,36,0.444444,AGAGAAACAGAAATATGAAACGGCCACCCCAGGTAT
14325,chrIV,14819,14854,TACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.31,chrIV:14819-14854,35,0.457143,AGAGAAACAGAAATATGAAACGGCCACCCCAGGTA
14326,chrIV,14820,14854,ACCTGGGGTGGCCGTTTCATATTTCTGTTTCTCT,42.75,chrIV:14820-14854,34,0.470588,AGAGAAACAGAAATATGAAACGGCCACCCCAGGT


### build a bowtie2 index from reference fasta

In [7]:
# build a bowtie2 index from the example FASTA
BT2_IDX = 'pipeline_output/bowtie2/example_idx'

idx_path = bowtie_build(input_file=EXAMPLE_FASTA, output_file=BT2_IDX)

print('DONE!', idx_path)

DONE! /home/conor/workspace/OligoMiner2/docs/website/notebooks/pipeline_output/bowtie2/example_idx


### build a jellyfish index from reference fasta

In [10]:
# build a jellyfish kmer index from the example FASTA
JF_FILE = 'pipeline_output/jellyfish/example_k18.jf'

jf_path = jellyfish_build(input_file=EXAMPLE_FASTA, output_file=JF_FILE, k=18)

print('DONE!', jf_path)

DONE! /home/conor/workspace/OligoMiner2/docs/website/notebooks/pipeline_output/jellyfish/example_k18.jf
